リスト7.1 必要なパッケージのインストール

In [ ]:
!pip install langgraph==0.4.9

リスト7.2 LangGraphの基本的なPythonコード例

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class GraphState(TypedDict):
    input: str    # 状態キー 'input' (変更なし)
    output: str

# ノード関数名は変更しても、しなくても良いですが、
# ノード名と合わせると分かりやすい場合もあります。
def process_initial_input(state: GraphState):
    processed_input = f"Processing input: {state['input']}" # 状態キー 'input' を参照
    return {"output": processed_input}

def generate_final_output(state: GraphState):
    final_output = f"Final output: {state['output']}"
    return {"output": final_output}

workflow = StateGraph(GraphState)

# ノードを追加 (ノード名を状態キーと異なる名前に変更)
workflow.add_node("input_processing_node", process_initial_input)  # 例: "input_processing_node" に変更
workflow.add_node("output_generation_node", generate_final_output) # 例: "output_generation_node" に変更

# エッジを追加 (変更後のノード名を使用)
workflow.add_edge("input_processing_node", "output_generation_node")

# エントリーポイントを設定 (変更後のノード名を使用)
workflow.set_entry_point("input_processing_node")

# 終了点を設定 (変更後のノード名を使用)
workflow.add_edge("output_generation_node", END)

# グラフをコンパイル
graph = workflow.compile()

# 実行 (入力辞書のキーは GraphState の 'input' のまま)
inputs = {"input": "example data"}
result = graph.invoke(inputs)
print(result['output'])

# Mermaid形式でグラフを可視化するコード (省略)
# ...
# Mermaid形式でグラフを可視化する（オプションの依存関係が必要です）
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid visualization requires additional dependencies.")

リスト7.3 必要なパッケージのインストール

In [ ]:
!pip install langchain-google-genai==2.1.5

リスト7.4 LangMemの基本的なPythonコード例

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
import asyncio
import os
from dotenv import load_dotenv
load_dotenv()

# API Keyの設定
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

# シンプルなメモリ管理クラス
class SimpleMemory:
    def __init__(self):
        self.memory = {}

    def store(self, key, value):
        """メモリに情報を保存する"""
        self.memory[key] = value
        return f"保存しました: {key} = {value}"

    def retrieve(self, key):
        """メモリから情報を取得する"""
        if key in self.memory:
            return f"検索結果: {key} = {self.memory[key]}"
        else:
            return f"情報が見つかりませんでした: {key}"

    def search(self, query):
        """キーワードで検索する"""
        results = []
        for key, value in self.memory.items():
            if query.lower() in key.lower() or query.lower() in value.lower():
                results.append(f"{key} = {value}")

        if results:
            return f"検索結果: {', '.join(results)}"
        else:
            return "検索結果: 関連する情報は見つかりませんでした"

# メモリインスタンスの作成
memory = SimpleMemory()

# LLMの初期化
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
)

# システムプロンプト
system_prompt = """あなたは記憶能力を持つアシスタントです。ユーザーの情報を記憶し、後で思い出すことができます。

私はメモリ機能を組み込みました。情報を保存するには「保存: キー=値」と指示し、
情報を取得するには「取得: キー」と指示し、キーワードで検索するには「検索: キーワード」と指示してください。

例えば：
- ユーザーが「私の好きな色は青です」と言った場合、「保存: 好きな色=青」と指示してください
- ユーザーが「私の好きな色は何？」と聞いた場合、「取得: 好きな色」または「検索: 色」と指示してください

そして、メモリから返ってきた情報に基づいて適切に応答してください。
"""

# 会話履歴
conversation_history = []

# メモリコマンドを検出して実行する関数
def process_memory_commands(text):
    if text.startswith("保存:"):
        try:
            # 「保存: キー=値」形式のコマンドを処理
            content = text[3:].strip()
            key, value = content.split('=', 1)
            result = memory.store(key.strip(), value.strip())
            return result
        except Exception as e:
            return f"保存コマンドの実行中にエラーが発生しました: {str(e)}"

    elif text.startswith("取得:"):
        try:
            # 「取得: キー」形式のコマンドを処理
            key = text[3:].strip()
            result = memory.retrieve(key)
            return result
        except Exception as e:
            return f"取得コマンドの実行中にエラーが発生しました: {str(e)}"

    elif text.startswith("検索:"):
        try:
            # 「検索: キーワード」形式のコマンドを処理
            query = text[3:].strip()
            result = memory.search(query)
            return result
        except Exception as e:
            return f"検索コマンドの実行中にエラーが発生しました: {str(e)}"

    return None

# 会話を実行する関数
def run_conversation(user_input):
    global conversation_history

    # ユーザー入力を会話履歴に追加
    conversation_history.append(HumanMessage(content=user_input))

    # システムプロンプトを含むメッセージリストを作成
    messages = [HumanMessage(content=system_prompt)]
    messages.extend(conversation_history)

    # LLMで応答を生成
    ai_response = llm.invoke(messages)

    # 会話履歴にAI応答を追加
    conversation_history.append(ai_response)

    # メモリコマンドを検出して実行
    memory_command_result = None
    for line in ai_response.content.split('\n'):
        result = process_memory_commands(line.strip())
        if result:
            memory_command_result = result
            break

    # メモリコマンドの実行結果を会話履歴に追加（LLMに見せるため）
    if memory_command_result:
        feedback_message = HumanMessage(content=f"システム: {memory_command_result}")
        conversation_history.append(feedback_message)

    return ai_response.content, memory_command_result

# ステップ1: 好きな色についての情報を保存
print("\n=== 情報の保存 ===\n")
ai_response, memory_result = run_conversation("私の好きな色は青です。この情報を覚えておいてください。")
print(f"AI応答: {ai_response}")
if memory_result:
    print(f"メモリ操作結果: {memory_result}")

# ステップ2: 好きな色を尋ねる
print("\n=== 情報の検索 ===\n")
ai_response, memory_result = run_conversation("私の好きな色は何でしたか？")
print(f"AI応答: {ai_response}")
if memory_result:
    print(f"メモリ操作結果: {memory_result}")

# 最後にメモリの内容を表示
print("\n=== メモリの内容 ===\n")
for key, value in memory.memory.items():
    print(f"{key}: {value}")


=== 情報の保存 ===

AI応答: 保存: 好きな色=青
メモリ操作結果: 保存しました: 好きな色 = 青

=== 情報の検索 ===

AI応答: 取得: 好きな色
メモリ操作結果: 検索結果: 好きな色 = 青

=== メモリの内容 ===

好きな色: 青


リスト7.5 必要なパッケージのインストール

In [ ]:
!pip install langchain_openai==0.3.25

リスト7.6 LangSmithの基本的なPythonコード例

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from dotenv import load_dotenv
load_dotenv()



# LangSmith APIキーを設定する（環境変数に設定されていることを確認）
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv('LANGCHAIN_API_KEY')  # 実際のAPIキーに置き換えてください
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# LLMを初期化する
llm = ChatOpenAI()

# プロンプトテンプレートを作成する
prompt = PromptTemplate.from_template("What is the capital of {country}?")

# LLMChainを作成する
chain = LLMChain(llm=llm, prompt=prompt)

# トレーシングを有効にしてチェーンを実行する
result = chain.invoke({"country": "Japan"})
print(result)

/tmp/ipython-input-3792159045.py:21: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


{'country': 'Japan', 'text': 'The capital of Japan is Tokyo.'}


リスト7.7 LangSmithクライアントを使用してデータセットを作成する例

In [ ]:
# LangSmithクライアントを使用してデータセットを作成する例
from langsmith import Client

client = Client()
dataset_name = "example-langsmith-dataset"
inputs = [{"country": "USA"}, {"country": "France"}]
outputs = [{"text": "Washington, D.C."}, {"text": "Paris"}]

try:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="An example dataset for testing a capital city chain.",
    )
    client.create_examples(
        inputs=inputs,
        outputs=outputs,
        dataset_id=dataset.id,
    )
    print(f"Dataset '{dataset_name}' created successfully.")
except Exception as e:
    print(f"Error creating dataset: {e}")

リスト7.8 必要なパッケージのインストール

In [ ]:
!pip install grandalf==0.8

リスト7.9 LangGraph、LangMem、LangSmithを統合した質問応答システムの例

In [ ]:
import os
from typing import TypedDict, List, Annotated

# grandalf のインストールを促す (任意)
# pip install grandalf
try:
    import grandalf
except ImportError:
    print("警告: grandalf がインストールされていません。グラフのASCII表示はスキップされます。")
    print("インストールするには pip install grandalf を実行してください。")

from langchain_openai import ChatOpenAI
# BaseMessage と HumanMessage をインポート
from langchain_core.messages import BaseMessage, HumanMessage
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# --- APIキー設定 ---
# (省略: 前回のコードと同じAPIキー設定部分をここに記述してください)
from dotenv import load_dotenv
load_dotenv()
LANGCHAIN_API_KEY = os.getenv('LANGCHAIN_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not LANGCHAIN_API_KEY:
    print("警告: LangChain APIキーが設定されていません。")
    # 必要に応じてデフォルトキーやエラー処理を追加
if not OPENAI_API_KEY:
    print("警告: OpenAI APIキーが設定されていません。")
    # 必要に応じてデフォルトキーやエラー処理を追加

os.environ["LANGCHAIN_TRACING_V2"] = "true"
if LANGCHAIN_API_KEY:
    os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY # OpenAI APIキーも設定

# LLMを初期化する
llm = ChatOpenAI(model_name="gpt-4o-mini")

# LangGraphの状態を定義する
class QASystemState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    # knowledge は存在しない場合もあるため Optional (または None 許容) にする
    knowledge: str | None
    answer: str

# 質問ルーティング関数 (変更なし、条件分岐用に文字列を返す)
def route_question(state: QASystemState):
    """質問の内容に基づいて、知識検索が必要か、直接回答生成に進むかを判断する"""
    print("--- ルーティング判断 ---")
    # state["messages"] には BaseMessage オブジェクトのリストが入る
    last_message = state["messages"][-1]
    # content 属性にアクセス
    if "知識" in last_message.content:
        print(f"メッセージ内容「{last_message.content}」に「知識」が含まれるため、知識検索へ")
        return "search_knowledge"
    else:
        print(f"メッセージ内容「{last_message.content}」に「知識」が含まれないため、直接回答生成へ")
        return "generate_answer"

# 知識検索ノード (変更なし、状態更新のため辞書を返す)
def search_knowledge(state: QASystemState):
    """知識を検索する (この例ではダミーの知識を返す)"""
    print("--- 知識検索ノード ---")
    last_message_content = state["messages"][-1].content
    print(f"最後のメッセージに基づいて知識を検索: {last_message_content}")
    searched_knowledge = f"「{last_message_content}」に関連すると仮定したダミーの知識です。"
    # 状態を更新する辞書を返す
    return {"knowledge": searched_knowledge}

# 回答生成ノード (変更なし、状態更新のため辞書を返す)
def generate_answer(state: QASystemState):
    """LLMを使用して回答を生成する"""
    print("--- 回答生成ノード ---")
    # knowledge が None の可能性を考慮
    knowledge = state.get("knowledge") # get() を使うとキーが存在しなくてもエラーにならない

    prompt_messages = [
        ("system", "あなたは親切なアシスタントです。提供された知識があれば、それを参考にして回答してください。"),
        state["messages"][-1], # 最新のメッセージ
    ]
    # 知識がある場合のみプロンプトに追加
    if knowledge:
        prompt_messages.append(
            ("system", f"参考知識:\n{knowledge}\n\n上記を踏まえて回答してください。")
        )
    else:
         # 知識がない場合の指示を追加（任意）
         prompt_messages.append(
             ("system", "今回は参考知識はありません。一般的な知識で回答してください。")
         )


    prompt = ChatPromptTemplate.from_messages(prompt_messages)
    chain = prompt | llm | StrOutputParser()

    print("LLMに回答生成を依頼中...")
    # chain.invoke に渡す辞書はプロンプト変数がなければ空で良い
    result = chain.invoke({})

    # 状態を更新する辞書を返す
    return {"answer": result}

# --- LangGraph構築の修正 ---
graph_builder = StateGraph(QASystemState)

# 状態を更新するノードを追加 (route_question はノードとして追加しない)
graph_builder.add_node("search_knowledge", search_knowledge)
graph_builder.add_node("generate_answer", generate_answer)

# ★★★ 修正点: 条件付きエントリーポイントを設定 ★★★
# グラフ開始時に route_question を実行し、その結果で分岐する
graph_builder.set_conditional_entry_point(
    route_question, # 条件判断を行う関数
    {
        # 関数の戻り値と次に遷移するノード名のマッピング
        "search_knowledge": "search_knowledge",
        "generate_answer": "generate_answer"
    }
)

# 通常のエッジを設定
graph_builder.add_edge("search_knowledge", "generate_answer") # 知識検索の後は回答生成へ
graph_builder.add_edge("generate_answer", END) # 回答生成の後は終了

# グラフをコンパイル
graph = graph_builder.compile()

# グラフの構造をASCIIアートで表示（grandalf がインストールされていれば）
try:
    graph.get_graph().print_ascii()
except ImportError:
    print("\nASCIIグラフ表示をスキップしました (grandalf未インストール)。")
except Exception as e:
    print(f"グラフのASCII表示に失敗しました: {e}")


# --- 実行 ---
print("\n--- 実行例1: 知識検索が必要なケース ---")
# ★★★ 修正点: 入力を HumanMessage オブジェクトのリストにする ★★★
inputs1 = {"messages": [HumanMessage(content="LangGraphの知識について教えてください。")]}
try:
    print("\nグラフ実行 (invoke):")
    result1 = graph.invoke(inputs1)
    print("\n--- 最終結果1 ---")
    # result1 は最終状態の辞書
    print(f"最終状態: {result1}")
    print(f"回答: {result1.get('answer', '回答が生成されませんでした')}")

    # ストリームで実行する場合 (デバッグに便利)
    # print("\nグラフ実行 (stream):")
    # final_state = None
    # for chunk in graph.stream(inputs1):
    #     print(chunk)
    #     print("---")
    #     final_state = list(chunk.values())[0] # 各ステップの状態を取得
    # print("\n--- 最終状態 (Stream) ---")
    # print(final_state)
    # if final_state and 'answer' in final_state:
    #      print(f"回答: {final_state['answer']}")

except Exception as e:
    print(f"\nエラーが発生しました: {e}")
    import traceback
    traceback.print_exc() # 詳細なトレースバックを表示

print("\n" + "="*30 + "\n")

# 実行例2: 知識検索が不要なケース
print("\n--- 実行例2: 知識検索が不要なケース ---")
inputs2 = {"messages": [HumanMessage(content="こんにちは、調子はどう？")]}
try:
    print("\nグラフ実行 (invoke):")
    result2 = graph.invoke(inputs2)
    print("\n--- 最終結果2 ---")
    print(f"最終状態: {result2}")
    print(f"回答: {result2.get('answer', '回答が生成されませんでした')}")
except Exception as e:
    print(f"\nエラーが発生しました: {e}")
    import traceback
    traceback.print_exc()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.5 MB/s eta 0:00:00
                   +-----------+           
                   | __start__ |           
                   +-----------+           
                 ...            ...        
               ..                  ..      
             ..                      ..    
+------------------+                   ..  
| search_knowledge |                 ..    
+------------------+               ..      
                 ***            ...        
                    **        ..           
                      **    ..             
                +-----------------+        
                | generate_answer |        
                +-----------------+        
                         *                 
                         *                 
                         *                 
                    +---------+            
                    | __end__ |            
                    +---------+           